In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

In [2]:
factor = 1000

In [3]:
def myfunc_w(X, y, power=1.0, eps=1e-6):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42,
    )

    X_train_mat = X_train.to_numpy(dtype=float)
    X_test_mat  = X_test.to_numpy(dtype=float)
    ys = y_train.to_numpy(dtype=float)

    max_j = np.max(X_train_mat, axis=0)

    normalized = X_train_mat / max_j
    sum_norm = np.sum(normalized, axis=1)+eps

    weights = 1.0 / (sum_norm ** power)

    A = np.hstack([X_train_mat, np.ones((X_train_mat.shape[0], 1))])

    W = weights[:, None]
    A_w = W * A
    y_w = W.ravel() * ys

    beta, *_ = np.linalg.lstsq(A_w, y_w, rcond=None)

    A_test = np.hstack([X_test_mat, np.ones((X_test_mat.shape[0], 1))])
    y_pred = A_test @ beta

    r2 = float(r2_score(y_test.to_numpy(dtype=float), y_pred))

    return beta, r2

In [4]:
def run_myfunc_w(df, target="ns", power=1.0, drop_columns=None, handler=None):
    if handler is not None:
        df = handler(df)

    if drop_columns is None:
        drop_columns = []

    X = df.drop(columns=[target] + drop_columns)
    y = df[target]

    beta, r2 = myfunc_w(X, y, power=power)
    return np.int64(np.round(beta*factor)), r2

In [5]:
df = pd.read_csv('benchmark_results/APPEND.csv')
run_myfunc_w(df)

(array([  97,  192, 2715]), 0.9984791193455083)

In [6]:
df = pd.read_csv('benchmark_results/CAT.csv')
run_myfunc_w(df)

(array([   8, 2706]), 0.9966775084118872)

In [7]:
df = pd.read_csv('benchmark_results/CLEAR.csv')
run_myfunc_w(df, power=3.0)

(array([  93,  -15, 1515]), 0.9865534456599984)

In [8]:
df = pd.read_csv('benchmark_results/CLEARITEMS.csv')
run_myfunc_w(df)

(array([ 103, 1455]), 0.9923474635352815)

In [9]:
df = pd.read_csv('benchmark_results/CONVERT.csv')
print(run_myfunc_w(df, handler=lambda df: df[df['type'] == 'Array'], drop_columns=['type']))
df = pd.read_csv('benchmark_results/CONVERT.csv')
print(run_myfunc_w(df, handler=lambda df: df[df['type'] == 'ByteString'], drop_columns=['type']))

(array([ 318, 3435]), 0.9995580137302795)
(array([   7, 3417]), 0.9948732866122838)


In [10]:
df = pd.read_csv('benchmark_results/DROP.csv')
run_myfunc_w(df)

(array([  99, 1486]), 0.9999616583290638)

In [11]:
df = pd.read_csv('benchmark_results/DUP.csv')
run_myfunc_w(df)

(array([   7, 3142]), 0.990259081583946)

In [12]:
df = pd.read_csv('benchmark_results/HASKEY.csv')
run_myfunc_w(df)

(array([  99, 2575]), 0.9999178734882064)

In [13]:
df = pd.read_csv('benchmark_results/INITSLOT.csv')
run_myfunc_w(df)

(array([ 155, 2156]), 0.9979267653940074)

In [33]:
df = pd.read_csv('benchmark_results/ISNULL.csv')
run_myfunc_w(df)

(array([  97, 1236]), 0.9998926474618235)

In [34]:
df = pd.read_csv('benchmark_results/ISTYPE.csv')
run_myfunc_w(df)

(array([  97, 1154]), 0.9998958761752025)

In [14]:
df = pd.read_csv('benchmark_results/KEYS.csv')
run_myfunc_w(df)

(array([1419, 2606]), 0.9999637336666407)

In [15]:
df = pd.read_csv('benchmark_results/MEMCPY.csv')
run_myfunc_w(df)

(array([   1, 3514]), 0.9976381810259952)

In [16]:
# у них должны быть одинаковые свободные члены
df = pd.read_csv('benchmark_results/NEWARRAYT.csv')
print(run_myfunc_w(df, handler=lambda df: df[df['type'] == 'Any'], drop_columns=['type']))
df = pd.read_csv('benchmark_results/NEWARRAYT.csv')
print(run_myfunc_w(df, handler=lambda df: df[df['type'] == 'ByteString'], drop_columns=['type']))

(array([ 126, 2718]), 0.9859870314777375)
(array([ 849, 2721]), 0.9999135249202911)


In [17]:
df = pd.read_csv('benchmark_results/NEWBUFFER.csv')
run_myfunc_w(df)

(array([   6, 2507]), 0.9842623417216293)

In [18]:
df = pd.read_csv('benchmark_results/PACK.csv')
run_myfunc_w(df)

(array([ 173, 2649]), 0.9869225328768629)

In [19]:
df = pd.read_csv('benchmark_results/PACKMAP.csv')
run_myfunc_w(df, power=3.5)

(array([  80, 5281, 3481]), 0.9995953894669359)

In [20]:
df = pd.read_csv('benchmark_results/PICKITEM.csv')
run_myfunc_w(df, power=3.0)

(array([  91,    6, 2751]), 0.9826647731124007)

In [21]:
df = pd.read_csv('benchmark_results/POPITEM.csv')
run_myfunc_w(df, power=1.5)

(array([  91, 3078]), 0.9709481136643807)

In [22]:
# некорректные свободные члены
df = pd.read_csv('benchmark_results/REMOVE.csv')
print(run_myfunc_w(df, handler=lambda df: df[df['type'] == 'Array'], drop_columns=['type']))
df = pd.read_csv('benchmark_results/REMOVE.csv')
print(run_myfunc_w(df, handler=lambda df: df[df['type'] == 'Map'], drop_columns=['type']))

(array([  98,    9, 1991]), 0.9997662710789825)
(array([  96,  706, 6776]), 0.9856757072755089)


In [37]:
df = pd.read_csv('benchmark_results/REVERSEITEMS.csv')
print(run_myfunc_w(df, handler=lambda df: df[df['type'] == 'Array'], drop_columns=['type']))
df = pd.read_csv('benchmark_results/REVERSEITEMS.csv')
print(run_myfunc_w(df, handler=lambda df: df[df['type'] == 'Buffer'], drop_columns=['type', 'refsdelta']))

(array([  98,   19, 2043]), 0.9999718525695899)
(array([   9, 1690]), 0.999951222159008)


In [24]:
df = pd.read_csv('benchmark_results/REVERSEN.csv')
run_myfunc_w(df)

(array([  19, 1702]), 0.998928783137599)

In [25]:
df = pd.read_csv('benchmark_results/ROLL.csv')
run_myfunc_w(df)

(array([   5, 1910]), 0.8739258349860559)

In [26]:
df = pd.read_csv('benchmark_results/SETITEM.csv')
run_myfunc_w(df)

(array([  99,  350, 2942]), 0.9999171644473398)

In [27]:
df = pd.read_csv('benchmark_results/SIZE.csv')
run_myfunc_w(df)

(array([ 100, 2693]), 0.9996362207138569)

In [28]:
df = pd.read_csv('benchmark_results/ST.csv')
run_myfunc_w(df)

(array([  98, 1599]), 0.9999450035176572)

In [29]:
df = pd.read_csv('benchmark_results/SUBSTR.csv')
run_myfunc_w(df)

(array([   7, 2908]), 0.9922615244242385)

In [30]:
df = pd.read_csv('benchmark_results/UNPACK.csv')
run_myfunc_w(df)

(array([ 254, 2604]), 0.9745961081466958)

In [31]:
# некорректный свободный член
df = pd.read_csv('benchmark_results/VALUES.csv')
run_myfunc_w(df, power=6.0)

(array([ 307,  369, 9868]), 0.992412011263309)

In [32]:
df = pd.read_csv('benchmark_results/XDROP.csv')
run_myfunc_w(df)

(array([  98,    6, 1791]), 0.9999667583238937)